# Protocol 2 — Active Inference Agent Simulations

Forward simulations for Protocol 2 (*Active Inference Agent Simulations: Adaptive Advantages of Somatic
Markers*), updated to **protocol specification v1.1.0**. Seven architectures play an Iowa Gambling Task
(IGT) whose outcome noise follows the registered volatility schedule.

* **Pred 2.A** — Full APGI converges within 50–80 trials (necessary) and earns a reward ratio
  APGI / β_SM-lesion > 1.15 (permutation p < 0.01). *Falsified*: convergence outside 50–80, or not faster
  than GNWT-only (Mann–Whitney p ≥ 0.01).
* **Pred 2.B** — 70–85 % of ignition events are interoceptively dominant (Πⁱ·|zⁱ| > Πᵉ·|zᵉ|), above the 60 %
  floor (binomial p < 0.01). *Falsified*: < 60 %.
* **Pred 2.C** — Somatic-marker retrieval M̂ precedes threshold crossing: cross-correlation peak at negative lag
  (±5 trials), M̂ leads by ≥ 1 trial in ≥ 75 % of ignition events, and median within-seed Kendall τ > 0.3
  with a seed-level bootstrap 95 % CI.
* **Pred 2.D** — β_SM lesion degrades reward more than the Πⁱ-constant and α_M lesions, with the largest
  deficit at σ_env = 0.6 (permutation p < 0.01, 10 000 permutations).
* **Pred 2.E** — APGI BIC beats Standard PP and GNWT-only by ΔBIC ≥ 10 on **held-out human IGT data**
  (N = 80). Human-data fitting is separate from these forward simulations, which compute no BIC.
* **Pred 2.F–2.G** *(exploratory, no falsifying weight)* — Ignition-driven strategy change; computational
  critical-slowing proxy for Pred 6.C.

> Computational protocol only; no human participants are simulated. As the v1.1 JSON notes, results from
> Pred 2.B and 2.C are properties of the implemented model under a frozen environment, not independent
> empirical confirmation. CP-3a calibration (independent N ≥ 60) and the scaled information-matrix pre-check
> must pass before any confirmatory run.
>
> **EP-2 | Paper 1 | Computational | Depends on: — | Spec v1.1.0 | 1000 seeds per condition (minimum 500); human IGT test N = 80**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| Architectures | 5 agents | 7: APGI-full, GNWT-only, Standard PP, β_SM-lesion, Πⁱ-constant, α_M-lesion, random |
| Volatility | Separate σ_env runs | Fixed within-run schedule 0.1 → 0.3 → 0.6 → 0.3 → 0.1 (5 × 100 trials) |
| Seeds | Single RNG, 100 agents | Deterministic `condition_index × 1000 + replicate_index`, 1000 per condition |
| Pred 2.A | Reward bars only | Convergence 50–80 trials + reward ratio > 1.15, α = 0.01 |
| Pred 2.C | — | Median within-seed Kendall τ > 0.3, ±5-trial cross-correlation, seed-level bootstrap |
| Pred 2.E | Simulated | Human-data BIC only (N = 80, disjoint from CP-3a calibration) |
| κ, α, β | 100, 0.3, 0.5 | κ = `null`; no legacy α/β defaults |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import binomtest, kendalltau, mannwhitneyu

from apgi.core import (
    GAMMA_SIG_DEFAULT,
    compute_pi_i_eff,
    ignition_probability,
    step_theta,
    theta_equilibrium,
)

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_2_somatic_agent_sim.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

## 1 — Protocol 2 parameters (v1.1)

The volatility schedule, seed formula and CP-3a somatic-marker weights come from the JSON. The registered
study runs 1000 seeds per condition; this notebook runs `N_SEEDS_DEMO` per architecture to stay fast.

In [ ]:
BLOCKS = PROTO["blocks"]
REPRO = PROTO["reproducibility"]
VOLATILITY = BLOCKS["volatility_schedule"]     # [0.1, 0.3, 0.6, 0.3, 0.1]
TRIALS_PER_BLOCK = BLOCKS["trials_per_block"]  # 100
N_TRIALS = BLOCKS["total_experimental_trials"] # 500
N_SEEDS_REGISTERED = PROTO["participants"]["n_target"]  # 1000 per condition
N_SEEDS_DEMO = 100

GAMMA_V = REPRO["cp3a"]["gamma_V_theoretical"]   # 0.6 (CP-3a theoretical default)
GAMMA_A = REPRO["cp3a"]["gamma_A_SM_theoretical"]  # 0.3
BETA_SM = 0.6                                     # ∈ typical range [0.3, 0.7]
KAPPA = PROTO["apgi_parameters"]["kappa"]["value"]  # None — uncalibrated
GAMMA_SIG = GAMMA_SIG_DEFAULT

# Seven architectures; feature flags = (somatic bias, ignition, adaptive Πⁱ, metabolic cost term, precision weighting)
ARCHITECTURES = {
    "apgi_full":      (1, 1, 1, 1, 1),
    "gnwt_only":      (0, 1, 0, 1, 0),
    "standard_pp":    (0, 0, 0, 0, 1),
    "beta_sm_lesion": (0, 1, 1, 1, 1),
    "pi_i_constant":  (1, 1, 0, 1, 1),
    "alpha_m_lesion": (1, 1, 1, 0, 1),
    "random":         (0, 0, 0, 0, 0),
}
LABELS = {"apgi_full": "APGI-full", "gnwt_only": "GNWT-only", "standard_pp": "Standard PP",
          "beta_sm_lesion": "β_SM-lesion", "pi_i_constant": "Πⁱ-constant", "alpha_m_lesion": "α_M-lesion",
          "random": "Random"}

# IGT decks A–D (payoffs / 100): A, B net −0.25 per trial; C, D net +0.25
GAIN = np.array([1.0, 1.0, 0.5, 0.5])
P_LOSS = np.array([0.5, 0.1, 0.5, 0.1])
LOSS = np.array([2.5, 12.5, 0.5, 2.5])

# Illustrative agent settings (synthetic; not registered)
LR_BASE, LR_IGNITION = 0.008, 0.12  # learning rate without / with ignition
INV_TEMP = 2.5
Z_I_SCALE = 1.2
ETA, RHO, PI_I_OPT = 0.02, 0.05, 1.0  # precision adaptation dΠⁱ = η(|εⁱ|−Πⁱ) + ρ(Πⁱ_opt−Πⁱ)
C_THETA, DELTA_INFO = 0.25, 0.10

print(f"Schedule {VOLATILITY} × {TRIALS_PER_BLOCK} trials; seed = {REPRO['seed_formula']}")
print(f"Running {N_SEEDS_DEMO} seeds per architecture (registered: {N_SEEDS_REGISTERED})")

## 2 — Agent model

The somatic marker $\hat M(c,a) = \gamma_V V(c,a) + \gamma_A A(c,a)$ uses learned deck valence $V$ and aversive
arousal $A$ (here entered with a negative sign). Marker retrieval biases choice through β_SM and sets
$\Pi^i_{\text{eff}} = \Pi^i \cdot e^{\beta_{SM} |\hat M|}$. At trial resolution, the gain set by a retrieval
takes effect on the following trial; the registered 10 ms simulation step would resolve within-trial ordering.
Ignition (sampled from $\sigma(\gamma_{sig}(S_t-\theta_t))$) triggers a deliberate, high-learning-rate update.

In [ ]:
def run_agent(arch, seed):
    rng = np.random.default_rng(seed)
    somatic, ignition_on, adaptive, cost, precision = ARCHITECTURES[arch]
    V, A = np.zeros(4), np.zeros(4)
    pi_i, theta, m_prev = 1.0, theta_equilibrium(0.5, 0.5), 0.0
    choice = np.zeros(N_TRIALS, int)
    reward = np.zeros(N_TRIALS)
    ignited = np.zeros(N_TRIALS, int)
    marker = np.zeros(N_TRIALS)
    dominant = []
    for t in range(N_TRIALS):
        sigma_env = VOLATILITY[t // TRIALS_PER_BLOCK]
        M = GAMMA_V * V - GAMMA_A * A
        if arch == "random":
            d = rng.integers(4)
        else:
            v = INV_TEMP * (V + BETA_SM * M * somatic)
            p = np.exp(v - v.max())
            d = rng.choice(4, p=p / p.sum())
        loss = LOSS[d] if rng.random() < P_LOSS[d] else 0.0
        r = GAIN[d] - loss + rng.normal(0, sigma_env)
        choice[t], reward[t] = d, r
        if arch == "random":
            continue
        marker[t] = abs(M[d])
        z_e = (r + loss) - GAIN[d]          # exteroceptive (gain-channel) error
        z_i = Z_I_SCALE * (loss - A[d])     # interoceptive (aversive-arousal) error
        pi_i_eff = compute_pi_i_eff(pi_i, BETA_SM * somatic, m_prev) if precision else 1.0
        S = abs(z_e) + pi_i_eff * abs(z_i)
        fired = bool(ignition_on) and rng.random() < ignition_probability(S, theta, GAMMA_SIG)
        lr = LR_IGNITION if fired else (2 * LR_BASE if arch == "standard_pp" else LR_BASE)
        V[d] += lr * (r - V[d])
        A[d] += lr * (loss - A[d])
        theta = step_theta(theta, C_t=float(fired), I_t=abs(z_i), fired=fired,
                           kappa_meta=C_THETA * cost, delta_info=DELTA_INFO)
        if adaptive:
            pi_i = float(np.clip(pi_i + ETA * (abs(z_i) - pi_i) + RHO * (PI_I_OPT - pi_i), 0.01, 10.0))
        if fired:
            ignited[t] = 1
            dominant.append(pi_i_eff * abs(z_i) > abs(z_e))
        m_prev = marker[t]
    return dict(choice=choice, reward=reward, ignited=ignited, marker=marker, dominant=np.array(dominant, bool))


def convergence_trial(choice, window=25, criterion=0.85, hold=50, floor=0.70):
    """First trial after which good-deck (C/D) choices stay high. Illustrative; the manifest locks the definition."""
    good = np.convolve((choice >= 2).astype(float), np.ones(window) / window, "valid")
    for t in range(len(good) - hold):
        if good[t] >= criterion and good[t:t + hold].min() >= floor:
            return t + window
    return np.nan


runs = {}
for cond_idx, arch in enumerate(ARCHITECTURES):
    runs[arch] = [run_agent(arch, cond_idx * 1000 + rep) for rep in range(N_SEEDS_DEMO)]

conv = {a: np.array([convergence_trial(r["choice"]) for r in runs[a]]) for a in runs}
cum_reward = {a: np.array([r["reward"].sum() for r in runs[a]]) for a in runs}
for a in runs:
    print(f"{LABELS[a]:<13} median convergence = {np.nanmedian(conv[a]) if np.isfinite(conv[a]).any() else float('nan'):6.1f} "
          f"(not converged {np.isnan(conv[a]).mean():.0%})   cumulative reward = {cum_reward[a].mean():7.1f}")

## 3 — Prediction 2.A: Convergence and reward advantage

In [ ]:
RNG_PERM = np.random.default_rng(REPRO["human_mcmc_seed"])
N_PERM = PRED["Pred 2.D"]["confirming_thresholds"]["permutations"]  # 10 000


def perm_test(x, y, alternative="greater"):
    """Two-sample permutation test on the difference of means."""
    pooled = np.concatenate([x, y])
    obs = x.mean() - y.mean()
    idx = np.argsort(RNG_PERM.random((N_PERM, len(pooled))), axis=1)
    perm = pooled[idx]
    diffs = perm[:, :len(x)].mean(1) - perm[:, len(x):].mean(1)
    if alternative == "less":
        diffs, obs = -diffs, -obs
    return (np.sum(diffs >= obs) + 1) / (N_PERM + 1)


THR_A = PRED["Pred 2.A"]["confirming_thresholds"]
lo, hi = THR_A["convergence_trials"]
conv_apgi = np.nanmedian(conv["apgi_full"])
assert cum_reward["beta_sm_lesion"].mean() > 0, "reward ratio needs a positive denominator"
ratio = cum_reward["apgi_full"].mean() / cum_reward["beta_sm_lesion"].mean()
p_ratio = perm_test(cum_reward["apgi_full"], cum_reward["beta_sm_lesion"])
gnwt_conv = np.where(np.isnan(conv["gnwt_only"]), N_TRIALS, conv["gnwt_only"])
apgi_conv = np.where(np.isnan(conv["apgi_full"]), N_TRIALS, conv["apgi_full"])
p_mw = mannwhitneyu(apgi_conv, gnwt_conv, alternative="less").pvalue

in_range = lo <= conv_apgi <= hi
confirmed_2A = in_range and ratio > THR_A["reward_ratio_strict_minimum"] and p_ratio < THR_A["permutation_alpha"]
falsified_2A = (not in_range) or p_mw >= 0.01
print(f"APGI median convergence = {conv_apgi:.1f} trials (registered {lo}–{hi}); "
      f"faster than GNWT-only: Mann–Whitney p = {p_mw:.2e}")
print(f"Reward ratio APGI / β_SM-lesion = {ratio:.2f}, permutation p = {p_ratio:.4f}")
print(f"Pred 2.A → {verdict(confirmed_2A, falsified_2A)}")

## 4 — Prediction 2.B: Interoceptive dominance of ignition events

In [ ]:
THR_B = PRED["Pred 2.B"]["confirming_thresholds"]
dom = np.concatenate([r["dominant"] for r in runs["apgi_full"]])
frac = dom.mean()
p_floor = binomtest(int(dom.sum()), len(dom), THR_B["floor"], alternative="greater").pvalue
lo_b, hi_b = THR_B["dominance_fraction_range"]
confirmed_2B = lo_b <= frac <= hi_b and p_floor < THR_B["binomial_alpha"]
falsified_2B = frac < THR_B["floor"]
print(f"{len(dom)} ignition events; interoceptively dominant = {frac:.1%} "
      f"(registered {lo_b:.0%}–{hi_b:.0%}; binomial vs {THR_B['floor']:.0%}: p = {p_floor:.1e})")
print(f"Pred 2.B → {verdict(confirmed_2B, falsified_2B)}")

## 5 — Prediction 2.C: Somatic marker precedes threshold crossing

Within each seed, cross-correlate M̂ with the ignition series $B_t$ over ±5 trials, take the peak lag, and
compute Kendall τ at that lag. Ignition events within a seed are not independent replicates, so inference
uses the median within-seed τ with a seed-level bootstrap.

In [ ]:
THR_C = PRED["Pred 2.C"]["confirming_thresholds"]
LAGS = np.arange(-5, 6)  # negative lag: M̂ leads B_t


def lagged(m, b, k):
    if k < 0:
        return m[:k], b[-k:]
    if k > 0:
        return m[k:], b[:-k]
    return m, b


xcorr, peak_lag, tau_seed, lead_frac = [], [], [], []
for r in runs["apgi_full"]:
    m, b = r["marker"], r["ignited"]
    cc = np.array([np.corrcoef(*lagged(m, b, k))[0, 1] for k in LAGS])
    k_star = LAGS[np.nanargmax(np.abs(cc))]
    xcorr.append(cc)
    peak_lag.append(k_star)
    tau_seed.append(kendalltau(*lagged(m, b, k_star))[0])
    ev = np.flatnonzero(b)
    ev = ev[ev >= 1]
    lead_frac.append(np.mean(m[ev - 1] > np.median(m)))  # elevated M̂ on the preceding trial
xcorr, peak_lag, tau_seed = np.array(xcorr), np.array(peak_lag), np.array(tau_seed)

boot_rng = np.random.default_rng(5)
boot = np.array([np.median(boot_rng.choice(tau_seed, len(tau_seed))) for _ in range(5000)])
tau_med, (tau_lo, tau_hi) = np.median(tau_seed), np.percentile(boot, [2.5, 97.5])
lead_ok = np.median(peak_lag) <= -THR_C["lead_trials_minimum"]
confirmed_2C = (lead_ok and np.median(lead_frac) >= THR_C["events_fraction_minimum"]
                and tau_med > THR_C["kendall_tau_strict_minimum"])
falsified_2C = np.median(peak_lag) >= 0
print(f"Median peak lag = {np.median(peak_lag):+.0f} trials; events led by elevated M̂ = {np.median(lead_frac):.0%}")
print(f"Median within-seed Kendall τ = {tau_med:.2f} (seed bootstrap 95% CI {tau_lo:.2f}–{tau_hi:.2f})")
print(f"Pred 2.C → {verdict(confirmed_2C, falsified_2C)}")

**Reading the Pred 2.B / 2.C results.** These verdicts describe this illustrative toy agent, not the frozen
APGI implementation that the registered sweep will run. Here marker retrieval and the loss outcome that
drives ignition share a trial, and the chosen deck carries both a large |M̂| and the loss risk. The
cross-correlation therefore peaks at lag 0, and the one-trial gain carry-over contributes only a weak
negative-lag component. The demo shows how the v1.1 decision rules separate CONFIRMED, INDETERMINATE and
FALSIFIED outcomes. It provides no evidence about Pred 2.C itself.

## 6 — Prediction 2.D: β_SM-lesion specificity at high volatility

In [ ]:
THR_D = PRED["Pred 2.D"]["confirming_thresholds"]
hv_block = VOLATILITY.index(THR_D["high_volatility"])
sl = slice(hv_block * TRIALS_PER_BLOCK, (hv_block + 1) * TRIALS_PER_BLOCK)
block_reward = {a: np.array([r["reward"][sl].mean() for r in runs[a]]) for a in runs}

p_vs_pi = perm_test(block_reward["beta_sm_lesion"], block_reward["pi_i_constant"], alternative="less")
p_vs_alpha = perm_test(block_reward["beta_sm_lesion"], block_reward["alpha_m_lesion"], alternative="less")
confirmed_2D = max(p_vs_pi, p_vs_alpha) < THR_D["permutation_alpha"]
falsified_2D = block_reward["beta_sm_lesion"].mean() >= block_reward["pi_i_constant"].mean()

full = block_reward["apgi_full"].mean()
for a in ["beta_sm_lesion", "pi_i_constant", "alpha_m_lesion"]:
    print(f"  {LABELS[a]:<13} σ_env = 0.6 reward/trial = {block_reward[a].mean():.3f}  "
          f"deficit vs APGI = {block_reward[a].mean() - full:+.3f}")
print(f"β_SM < Πⁱ-constant p = {p_vs_pi:.4f};  β_SM < α_M p = {p_vs_alpha:.4f}")
print(f"Pred 2.D → {verdict(confirmed_2D, falsified_2D)}")

## 7 — Prediction 2.E: human-data model comparison (not simulated)

Pred 2.E is a held-out maximum-likelihood fit of APGI-full, GNWT-only and Standard PP to the N = 80 in-vivo
IGT cohort (ΔBIC ≥ 10 versus each, cross-validated log-likelihood p < 0.01). The CP-3a calibration dataset
(N ≥ 60) must be disjoint from that cohort. The framework falsifier F2 requires ΔBIC ≤ 0 versus the best
alternative across **all** registered datasets; a single-dataset failure is not automatically F2.

In [ ]:
p2e = PRED["Pred 2.E"]
print("Confirming:", p2e["confirming_evidence"])
print("F2 falsifier:", p2e["framework_falsifier"])
print("Human cohort:", PROTO["participants"]["human_igt_test_n_target"],
      "| calibration N ≥", PROTO["participants"]["independent_calibration_n_minimum"])
print("Identifiability pre-check:", REPRO["identifiability"]["maximum_absolute_off_diagonal"],
      "max |off-diagonal|; scaled condition number ≤", REPRO["identifiability"]["scaled_condition_number_maximum"])

## 8 — Visualise Pred 2.A–2.D

In [ ]:
archs = list(ARCHITECTURES)
colors = ["#2166ac", "#66CC99", "#00CC99", "#d6604d", "#9966FF", "#FFCC00", "#AAAAAA"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

ax = axes[0]
for a, col in zip(archs[:3] + ["beta_sm_lesion"], colors[:3] + [colors[3]]):
    good = np.mean([(r["choice"] >= 2).astype(float) for r in runs[a]], axis=0)
    ax.plot(np.convolve(good, np.ones(10) / 10, "valid"), color=col, lw=1.4, label=LABELS[a])
for b in range(1, len(VOLATILITY)):
    ax.axvline(b * TRIALS_PER_BLOCK, color="k", lw=0.5, alpha=0.3)
ax.axvspan(lo, hi, color="#2166ac", alpha=0.08, label="50–80 trial benchmark")
ax.set_xlabel("Trial")
ax.set_ylabel("P(good deck), 10-trial mean")
ax.set_title(f"Pred 2.A — learning curves\nσ_env schedule {VOLATILITY}", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
means = [cum_reward[a].mean() for a in archs]
sems = [cum_reward[a].std() / np.sqrt(N_SEEDS_DEMO) for a in archs]
ax.bar(range(len(archs)), means, yerr=sems, color=colors, alpha=0.85, edgecolor="white", capsize=3)
ax.set_xticks(range(len(archs)), [LABELS[a] for a in archs], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Cumulative reward (500 trials)")
ax.set_title(f"Pred 2.A — APGI / β_SM ratio = {ratio:.2f}", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
ax.plot(LAGS, np.nanmean(xcorr, 0), "o-", color="#2166ac")
ax.axvline(0, color="k", lw=0.6, alpha=0.4)
ax.set_xlabel("Lag (trials; negative = M̂ leads)")
ax.set_ylabel("corr(M̂, ignition)")
ax.set_title(f"Pred 2.C — median τ = {tau_med:.2f}", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[3]
lesions = ["apgi_full", "beta_sm_lesion", "pi_i_constant", "alpha_m_lesion"]
ax.bar(range(4), [block_reward[a].mean() for a in lesions],
       yerr=[block_reward[a].std() / np.sqrt(N_SEEDS_DEMO) for a in lesions],
       color=[colors[0], colors[3], colors[4], colors[5]], alpha=0.85, edgecolor="white", capsize=3)
ax.set_xticks(range(4), [LABELS[a] for a in lesions], rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Reward / trial at σ_env = 0.6")
ax.set_title("Pred 2.D — lesion specificity", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Protocol 2 — Somatic-marker agents · spec v1.1.0 · {N_SEEDS_DEMO} seeds/architecture (registered 1000)",
             fontsize=12)
plt.tight_layout()
plt.show()